# Field Soil Moisture Map

Visualizes **per-pixel** surface soil moisture (0–5 cm) inside a field boundary — not
the field mean — for a satellite acquisition date you choose interactively.

In this notebook you will:
- Load a field boundary from a vector file (GeoJSON, Shapefile, GeoPackage, ... — any
  format geopandas/OGR supports)
- Download clipped GeoTIFFs for the field via the MoistX file endpoint (cached locally),
  including the per-pixel sensors and quality flag files
- Extract every pixel's soil moisture value, quality flag and sensors for each acquisition date
- Pick a date from a dropdown and see that date's pixel-level soil moisture on a map,
  zoomed to the field, with an optional quality flag layer

Uses `POST /get/file/soil-moisture` — see [Swagger UI](https://moistx.com/api/docs) /
[ReDoc](https://moistx.com/api/redoc) for the full API reference, and the repo
[README](./README.md#data-license--attribution) for data licensing/attribution terms.

In [1]:
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from dotenv import load_dotenv
load_dotenv()

import geopandas as gpd
import pandas as pd
from IPython.display import display, update_display

from helpers import load_field_geometry, download_sm_files, load_pixel_timeseries
from plotting import plot_pixel_sm_map, build_sm_calendar

In [2]:
# ── Parameters — edit here ────────────────────────────────────────────────────

API_KEY = os.getenv('MOISTX_API_KEY')
if not API_KEY:
    raise RuntimeError(
        'MOISTX_API_KEY not set — add your key to .env '
        '(get one from https://moistx.com/dashboard)'
    )
BASE_URL  = 'https://moistx.com/api'
CACHE_DIR = './cache'

# Field boundary — any OGR-supported vector file (GeoJSON, Shapefile, GeoPackage, KML, ...)
FIELD_FILE = './example_field.geojson'

# Date range to fetch — max 365 days
START = '2025-05-01T00:00:00'
END   = '2025-09-30T23:59:59'

## Load field geometry

`load_field_geometry` reads any OGR-supported vector file and reprojects it to WGS84;
multiple features are unioned into a single field polygon.

In [3]:
field_geom, field_wkt = load_field_geometry(FIELD_FILE)
center = [field_geom.centroid.y, field_geom.centroid.x]

field_gdf = gpd.GeoDataFrame(geometry=[field_geom], crs='EPSG:4326')
area_ha = field_gdf.to_crs(field_gdf.estimate_utm_crs()).area.iloc[0] / 10_000
print(f'Field area: {area_ha:.1f} ha')

Field area: 4.7 ha


## Download soil moisture GeoTIFFs for the field

Same endpoint used in `02_irrigation_detection.ipynb`: `POST /get/file/soil-moisture`
returns a ZIP of GeoTIFFs clipped to the field polygon, one per acquisition date.
With `include_metadata=True` the ZIP also holds, per date, a **sensors** file (which
sensors contributed to each pixel) and a **quality flag** file (`good` = NDVI < 0.75,
`poor` = NDVI ≥ 0.75), where the region provides them.
Files are cached locally by (polygon + date range), so re-running skips the API call.

In [4]:
# include_metadata=True also downloads the per-pixel sensors and quality flag files
field_files = download_sm_files(
    field_wkt, START, END, CACHE_DIR, API_KEY, BASE_URL, label='field',
    include_metadata=True,
)

[field] Fetching 2025-05-01 → 2025-09-30 …
[field] 80 GeoTIFFs downloaded and cached (+ sensors/quality files for 0)


## Extract per-pixel soil moisture

Each pixel inside the field boundary keeps its own value per date — no averaging.
`sm_min`/`sm_max` summarize the whole season's range for reference; the map below
auto-scales its color to each individual date's own range so spatial patterns stay
visible even on days where the field-wide spread is only a couple of vol%.

In [5]:
field_pixels = load_pixel_timeseries(field_files, field_geom)

available_dates = sorted(field_pixels['datetime'].unique())
sm_min, sm_max   = field_pixels['sm'].min(), field_pixels['sm'].max()

print(f'{len(available_dates)} acquisition dates, {field_pixels["pixel_id"].nunique()} pixels')
print(f'Soil moisture range: {sm_min:.1f} – {sm_max:.1f} vol%')
if field_pixels['quality'].notna().any():
    print('Quality flags:', field_pixels['quality'].value_counts(normalize=True).round(2).to_dict())
else:
    print('Quality flags: not available for this region')

  Loaded 80/80 files …
59 acquisition dates, 119 pixels
Soil moisture range: 13.6 – 29.4 vol%
Quality flags: not available for this region


## Interactive map — pick a date from the calendar

Each day in the calendar below is colored by that date's field-mean soil moisture
(light = drier, dark blue = wetter), fixed to the season's overall vol% range so
color is comparable across months. Days with no acquisition are grayed out and
can't be clicked. Click an available day (or page with ◀ / ▶) to load that date's
per-pixel soil moisture on the map below, zoomed to the field — hovering a pixel
gives its exact value, quality flag and contributing sensors.

Where quality flags are available, the layer control (top right) also offers a
**Quality flag** layer: green = `good` (NDVI < 0.75), orange = `poor` (NDVI ≥ 0.75,
dense vegetation — SM is less reliable there).

In [ ]:
MAP_DISPLAY_ID = 'field_sm_map'

def _render_map(selected_dt):
    day_pixels = field_pixels[field_pixels['datetime'] == selected_dt]
    update_display(
        plot_pixel_sm_map(field_geom, day_pixels, center),
        display_id=MAP_DISPLAY_ID,
    )

initial_date = available_dates[-1]

calendar_widget = build_sm_calendar(
    field_pixels,
    sm_min=sm_min,
    sm_max=sm_max,
    initial_date=initial_date,
    on_select=_render_map,
)
display(calendar_widget)

initial_pixels = field_pixels[field_pixels['datetime'] == initial_date]
display(plot_pixel_sm_map(field_geom, initial_pixels, center), display_id=MAP_DISPLAY_ID)

<DisplayHandle display_id=field_sm_map>